<a href="https://colab.research.google.com/github/erizz2/PHYS220-Project/blob/main/PHYS220proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
# import useful libraries
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import math
from scipy.integrate import quad
from IPython.display import HTML

# functions
def f(x, t):
    return

def calculateBn(init, n, x, L):
    return (2 / L) * scipy.integrate.quad(init(x) * np.sin((n * np.pi * x) / L), 0, L)

def u(x, t, N, n, L, init):
    for n in range(1, N+1):
        u += calculateBn(init, n, x, L) * np.sin(n * np.pi * x / L) * np.exp(-a(n * np.pi / L)**2 * t)
    return u(x,t)

def explicit_method(k, h, BC1, BC2, N):

    '''
    k = time step
    h = spatial step
    T = Temperature
    N = number of points
    BC1 = left boundary condition
    BC2 = right boundary condition
    target = accuracy condition
    '''

    M = int(1000 / k)
    u = np.zeros([N+1, M+1], float)
    x = np.linspace(0, 100, N+1)

    gaussian = lambda x: 30000 * np.exp(-0.015*(x-50)**2)

    u[:,0] = gaussian(x) # initial spatial profile


    # boundary conditions
    u[0, :] = BC1
    u[N, :] = BC2

    #uprime = u.copy()
    r = k / h**2


    for n in range(M):
        for j in range(1, N):
            u[j, n+1] = (1 - 2*r)*u[j, n] + r*u[j-1, n] + r*u[j+1, n]

    return x, u

def implicit_method(k, h, BC1, BC2, N, target):

    '''
    k = time step
    h = spatial step
    T = Temperature
    N = number of points
    BC1 = left boundary condition
    BC2 = right boundary condition
    target = accuracy condition
    '''

    M = int(1000 / k)
    u = np.zeros([N+1, M+1], float)
    x = np.linspace(0, 100, N+1)

    gaussian = lambda x: 3000 * np.exp(-0.01*(x-50)**2)

    u[:,0] = gaussian(x) # initial spatial profile


    # boundary conditions
    u[0, :] = BC1
    u[N, :] = BC2

    #uprime = u.copy()
    r = k / h**2


    for n in range(M):
        u[:, n+1] = u[:, n]
        u[0, n+1] = BC1
        u[N, n+1] = BC2

        delta = 1.0

        while delta > target:

          for j in range(1, N):
            uprime = u[:, n+1].copy()

            u[j, n+1] = 1 / (1 + 2*r) * (u[j, n] + r*u[j-1, n+1] + r*u[j+1, n+1])
          delta = np.max(np.abs(uprime - u[:, n+1]))
          #uprime = u.copy()


    return x, u


def update_color_bar(frame):
    step = frame
    if step < phi.shape[1]:
        new_data = phi[:, step].reshape(1, -1)
        im.set_array(new_data)

        time_text.set_text(f"Timestep: {step}")

    return [im, time_text]

def update(frame):
    #print(frame)
    line.set_data(x, phi[:, frame])
    return line,

# main
if __name__ == "__main__":
    x, phi = explicit_method(1.0, 3.0, 0.0, 0.0, 100)

    """
    plt.figure()
    plt.imshow(phi, origin='lower', cmap='inferno')
    plt.colorbar(label='Temperature (K)')
    plt.ylabel('x (m)')
    plt.xlabel('t (s)')
    plt.title("Heat vs. Time")
    plt.show()
    """
    '''
    fig, ax = plt.subplots()
    line, = ax.plot([], [])
    ax.set_xlim(0, 100)
    ax.set_ylim(np.min(phi), np.max(phi))
    #x = np.linspace(0, 100, 101)
    #ani = FuncAnimation(fig, update, frames = range(0, phi.shape[1], 10), init_func=init, interval = 50, blit=False)
    ani = FuncAnimation(fig, update, frames = range(0, phi.shape[1], 10), interval = 50, blit=False)

    plt.close(fig)

    display(HTML(ani.to_jshtml()))
    '''

    fig, ax = plt.subplots(figsize=(10, 4))
    rod_data = phi[:, 0].reshape(1, -1)

    im = ax.imshow(rod_data, aspect=5.0, cmap='inferno')

    time_text = ax.text(0.5, 0.01, '', fontsize=12, color='white')
    ax.set_yticks([])
    ax.set_xlabel("Position (m)")
    ax.set_title("1D Heat Equation Evolution")
    plt.colorbar(im, orientation='horizontal', label='Temperature (K)', pad=0.3)

    ani = FuncAnimation(fig, update_color_bar, frames = range(0, phi.shape[1], 10), interval=35, blit=True, repeat=True)

    plt.close(fig)

    display(HTML(ani.to_jshtml()))
